# EVM Analysis with AI-enhanced Forecasting
Software Project Management — AI-enhanced EVM Forecasting variant (odd roll number).
Single-notebook pipeline: synthetic sprint data → Parts 1–3 classical EVM baseline → Part 4 Random Forest forecast vs baseline.
All values in $m. Run top to bottom (`Run All`). Outputs: `evm_table.csv`, `evm_dashboard.png`, `evm_forecast_comparison.png`, `report.md`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_STATE = 9
SPLIT = 8  # train sprints 1-8, predict 9-15

## Part 0 — Synthetic dataset (15 sprints: sprint_id, PV, EV, AC)
Narrative: sprints 1–4 on track, 5–8 troubled (scope creep / overruns), 9–12 recovery, 13–15 plateau with slight slip.

In [ ]:
pv = [0.090, 0.095, 0.100, 0.100, 0.105, 0.105, 0.110, 0.100,
      0.100, 0.105, 0.100, 0.095, 0.095, 0.100, 0.105]
ev = [0.088, 0.097, 0.102, 0.098, 0.090, 0.088, 0.095, 0.092,
      0.102, 0.108, 0.104, 0.098, 0.092, 0.096, 0.100]
ac = [0.090, 0.094, 0.100, 0.102, 0.108, 0.110, 0.112, 0.105,
      0.100, 0.102, 0.098, 0.096, 0.098, 0.102, 0.106]
df = pd.DataFrame({'sprint_id': range(1, 16), 'PV': pv, 'EV': ev, 'AC': ac})
for c in ['PV', 'EV', 'AC']:
    df[f'cum_{c}'] = df[c].cumsum()
BAC = df['cum_PV'].iloc[-1]
print(f'BAC (final cumulative PV) = ${BAC:.3f}m')
df.head(15)

## Part 1 — Variance Calculation (CV = EV − AC, SV = EV − PV)

In [ ]:
df['CV'] = df['cum_EV'] - df['cum_AC']
df['SV'] = df['cum_EV'] - df['cum_PV']
df['CPI'] = df['cum_EV'] / df['cum_AC']
df['SPI'] = df['cum_EV'] / df['cum_PV']
df['status'] = np.where(df['CV'] < 0, 'over budget', 'under budget') + ', ' + np.where(df['SV'] < 0, 'behind schedule', 'ahead/on schedule')
df[['sprint_id', 'cum_PV', 'cum_EV', 'cum_AC', 'CV', 'SV', 'status']].round(4)

## Part 2 — Performance Index Analysis (CPI = EV/AC, SPI = EV/PV)

In [ ]:
print(df[['sprint_id', 'CPI', 'SPI']].round(4).to_string(index=False))
fig, ax = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('EVM Performance Dashboard (cumulative, $m)', fontweight='bold')
ax[0, 0].plot(df['sprint_id'], df['cum_PV'], marker='o', label='PV')
ax[0, 0].plot(df['sprint_id'], df['cum_EV'], marker='s', label='EV')
ax[0, 0].plot(df['sprint_id'], df['cum_AC'], marker='^', label='AC')
ax[0, 0].set_title('S-Curve: PV / EV / AC'); ax[0, 0].set_xlabel('Sprint'); ax[0, 0].legend(); ax[0, 0].grid(alpha=0.3)
ax[0, 1].plot(df['sprint_id'], df['CPI'], marker='o', label='CPI')
ax[0, 1].plot(df['sprint_id'], df['SPI'], marker='s', label='SPI')
ax[0, 1].axhline(1.0, color='gray', linestyle='--')
ax[0, 1].set_title('CPI / SPI (1.0 = on track)'); ax[0, 1].set_xlabel('Sprint'); ax[0, 1].legend(); ax[0, 1].grid(alpha=0.3)
x = df['sprint_id'].to_numpy()
ax[1, 0].bar(x - 0.2, df['CV'], width=0.4, label='CV')
ax[1, 0].bar(x + 0.2, df['SV'], width=0.4, label='SV')
ax[1, 0].axhline(0, color='black', linewidth=0.8)
ax[1, 0].set_title('Variances (cumulative)'); ax[1, 0].set_xlabel('Sprint'); ax[1, 0].legend(); ax[1, 0].grid(alpha=0.3, axis='y')
ax[1, 1].axis('off')
last = df.iloc[-1]
ax[1, 1].text(0.05, 0.95, f"Final sprint 15: CV=${last['CV']:.3f}m, SV=${last['SV']:.3f}m\nCPI={last['CPI']:.3f}, SPI={last['SPI']:.3f}\nTrough sprint 8: CPI={df.loc[7, 'CPI']:.3f}, SPI={df.loc[7, 'SPI']:.3f}",
             va='top', family='monospace', bbox=dict(boxstyle='round', facecolor='whitesmoke'))
fig.tight_layout(); fig.savefig('evm_dashboard.png', dpi=150); plt.show()

## Part 3 — EVM Forecasting (EAC / ETC, two standard formulas)

In [ ]:
at = 8
row8 = df.loc[df['sprint_id'] == at].iloc[0]
AC8, EV8, CPI8, SPI8 = row8['cum_AC'], row8['cum_EV'], row8['CPI'], row8['SPI']
EAC_cpi = BAC / CPI8                 # typical: current efficiency continues
EAC_ac = AC8 + (BAC - EV8)           # atypical: future at planned rate
EAC_comp = AC8 + (BAC - EV8) / (CPI8 * SPI8)  # bonus composite
for name, eac in [('EAC = BAC/CPI', EAC_cpi), ('EAC = AC + (BAC-EV)', EAC_ac), ('EAC composite (bonus)', EAC_comp)]:
    print(f'{name}: EAC=${eac:.4f}m  ETC=${eac - AC8:.4f}m  VAC=${BAC - eac:.4f}m')
print(f"\nActual final AC = ${df['cum_AC'].iloc[-1]:.4f}m")

## Part 4 — AI-enhanced Forecasting (Random Forest train 1–8 → predict CPI/SPI 9–15)

In [ ]:
df['CPI_lag1'] = df['CPI'].shift(1).bfill()
df['SPI_lag1'] = df['SPI'].shift(1).bfill()
FEATURES = ['sprint_id', 'cum_PV', 'cum_EV', 'cum_AC', 'CV', 'SV', 'CPI_lag1', 'SPI_lag1']
train = df[df['sprint_id'] <= SPLIT].copy()
test = df[df['sprint_id'] > SPLIT].copy()
model = RandomForestRegressor(n_estimators=300, max_depth=4, random_state=RANDOM_STATE)
model.fit(train[FEATURES], train[['CPI', 'SPI']])
pred = model.predict(test[FEATURES])
test['CPI_pred'], test['SPI_pred'] = pred[:, 0], pred[:, 1]
test['CPI_trad'], test['SPI_trad'] = train.iloc[-1]['CPI'], train.iloc[-1]['SPI']  # flat EVM baseline
for m, col, pr in [('CPI', 'CPI', 'CPI_pred'), ('SPI', 'SPI', 'SPI_pred')]:
    trad_col = col + '_trad'
    print(f"{m}: RF MAE={mean_absolute_error(test[col], test[pr]):.4f} vs trad={mean_absolute_error(test[col], test[trad_col]):.4f} | "
          f"RF RMSE={np.sqrt(mean_squared_error(test[col], test[pr])):.4f} vs trad={np.sqrt(mean_squared_error(test[col], test[trad_col])):.4f}")
test[['sprint_id', 'CPI', 'CPI_trad', 'CPI_pred', 'SPI', 'SPI_trad', 'SPI_pred']].round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5), sharex=True)
fig.suptitle('AI (Random Forest) vs Traditional EVM Forecast — CPI / SPI (sprints 9-15)', fontweight='bold')
for i, (col, pr, tr, title) in enumerate([('CPI', 'CPI_pred', 'CPI_trad', 'Cost Performance Index (CPI)'), ('SPI', 'SPI_pred', 'SPI_trad', 'Schedule Performance Index (SPI)')]):
    ax[i].plot(df['sprint_id'], df[col], 'k-o', label='Actual', linewidth=2)
    ax[i].plot(test['sprint_id'], test[tr], 'r--s', label='Traditional EVM (flat @ sprint 8)')
    ax[i].plot(test['sprint_id'], test[pr], color='b', marker='^', linestyle=(0, (3, 1)), label='AI predicted (RF)')
    ax[i].axvline(SPLIT + 0.5, color='gray', linestyle=':')
    ax[i].axhline(1.0, color='gray', linestyle='--', linewidth=1, alpha=0.6)
    ax[i].set_title(title); ax[i].set_xlabel('Sprint'); ax[i].set_xticks(range(1, 16)); ax[i].legend(fontsize=8); ax[i].grid(alpha=0.3)
fig.tight_layout(); fig.savefig('evm_forecast_comparison.png', dpi=150); plt.show()

## Part 5 — Markdown report (also saved to `report.md`)

In [ ]:
df.to_csv('evm_table.csv', index=False)
report = open('report.md').read() if __import__('os').path.exists('report.md') else ''
print('evm_table.csv saved. Run evm_analysis.py to regenerate report.md + PNGs.')
print(df.round(4).to_string(index=False))